<a href="https://colab.research.google.com/github/arwendy123/Customer-Churn-Prediction-and-Analysis/blob/main/Customer_Churn_Prediction_in_Telecommunications_A_Machine_Learning_Approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reducing Customer Churn Through Data Analysis

## 1.) Problem Description

Customer churn is a major challenge for telecommunications companies because losing existing customers reduces recurring revenue and increases acquisition costs.

This project analyzes telecom customer data to identify the factors most associated with churn and to determine which customers should be prioritized for retention efforts.

### Business Questions
- Which customer groups have the highest churn rate?
- Does contract type affect churn?
- How does customer tenure relate to churn?
- Are higher charges associated with higher churn?
- Which customers should be targeted by retention campaigns?

### Objectives
- Measure the overall churn rate.
- Identify the strongest churn drivers.
- Segment high-risk customers.
- Provide retention recommendations based on the analysis.

The predictive model is included as a supporting tool, while the main focus of this project is generating actionable business insights.

## 2.) Setup and Libraries

In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
# Notebook configuration
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

This notebook is organized to support a business-oriented churn analysis. The primary focus is identifying high-risk customer segments and translating analytical findings into actionable retention recommendations.

## 3.) Data Overview

### 3.1 Load and Preview Dataset

Load the dataset and preview the first five records to understand the available variables and the overall structure of the data.

In [3]:
df = pd.read_csv('../data/telco_churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 3.2 Data Structure

Inspect the number of records, column names, data types, and non-null counts.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

**Observation:** The dataset contains 7,043 records and 21 columns. Most variables are categorical (`str`), while `SeniorCitizen` and `tenure` are stored as integers and `MonthlyCharges` is stored as a numeric (`float64`) variable. The `TotalCharges` column is still stored as a text (`str`) field and will require additional cleaning before numerical analysis.

### 3.3 Variable Type Summary

In [10]:
df.dtypes.value_counts()

str        18
int64       2
float64     1
Name: count, dtype: int64

**Observation:** The dataset is dominated by categorical variables (18 columns), while only 3 columns are numeric. This indicates that the churn analysis will primarily focus on comparing churn rates across customer segments, service subscriptions, contract types, and payment methods.

### 3.4 Quick Summary



In [5]:
print(f'Total customers : {df.shape[0]:,}')
print(f'Total features : {df.shape[1]}')
print(f'Churn rate : {(df["Churn"] == "Yes").mean():.2%}')

Total customers : 7,043
Total features : 21
Churn rate : 26.54%


**Observation:** The dataset contains 7,043 customers and 21 features. The churn rate is approximately 26.5%, meaning about one in four customers left the company.

### 3.5 Missing Values

Check for missing values across all columns.

In [6]:
missing = df.isnull().sum()
missing_df = pd.DataFrame({ 
    'Missing Values': missing, 
    'Percentage': (missing / len(df) * 100).round(2) 
})

missing_df[missing_df['Missing Values'] > 0]

,Missing Values,Percentage


**Observation:** No explicit missing values were detected using `isnull()`. However, the `TotalCharges` column may still contain blank strings that are not recognized as missing values.

### 3.6 Duplicate Records

In [7]:
duplicates = df.duplicated().sum()
duplicates

np.int64(0)

**Observation:** No duplicate records were found in the dataset.

### 3.7 Initial Observations

Based on the initial inspection:
- The dataset contains **7,043 customer records** and **21 features**.
- The target variable is **`Churn`**.
- The overall churn rate is approximately **26.5%**.
- Most variables are categorical and describe customer services, contract type, and payment method.
- `TotalCharges` requires additional cleaning before further analysis.

The dataset is largely clean and suitable for business-oriented exploratory analysis.

## 4.) Data Cleaning

### 4.1 Inspect `TotalCharges`

The `TotalCharges` column is currently stored as a text field. Check for blank strings before converting it to a numeric type.

In [11]:
blank_totalcharges = (df['TotalCharges'].str.strip() == '').sum()

print(f'Blank values in TotalCharges: {blank_totalcharges}')

Blank values in TotalCharges: 11


**Observation:** The `TotalCharges` column contains 11 blank string values that are not detected by `isnull()`. These values must be handled before converting the column to a numeric type.

### 4.2 Convert `TotalCharges` to Numeric

In [12]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(df['TotalCharges'].dtype)

float64


**Observation:** `TotalCharges` has been successfully converted to a numeric (`float64`) variable. Invalid blank strings were converted into missing values (`NaN`).

### 4.3 Recheck Missing Values

In [13]:
missing_totalcharges = df['TotalCharges'].isnull().sum()

print(f'Missing values in TotalCharges: {missing_totalcharges}')

Missing values in TotalCharges: 11


**Observation:** After conversion, 11 missing values were identified in `TotalCharges`.

### 4.4 Investigate Rows with Missing `TotalCharges`

Inspect the records with missing `TotalCharges` to understand the cause of the missing values.

In [14]:
df[df['TotalCharges'].isnull()].head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No


**Observation:** The customers with missing `TotalCharges` have `tenure = 0`, indicating that they are newly joined customers who have not yet accumulated any charges.

### 4.5 Handle Missing Values

Because only 11 out of 7,043 records are affected (approximately 0.16% of the dataset), the incomplete records will be removed.

In [15]:
df = df.dropna(subset=['TotalCharges'])

print(df.shape)

(7032, 21)


**Observation:** The 11 incomplete records were removed, leaving 7,032 rows available for analysis.

### 4.6 Create Numeric Target Variable

Create a numeric version of the churn target for future analysis and modeling.

In [16]:
df['Churn_Flag'] = df['Churn'].map({'No': 0, 'Yes': 1})

df[['Churn', 'Churn_Flag']].head()

,Churn,Churn_Flag
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


**Observation:** The original business-friendly `Churn` column was preserved, while `Churn_Flag` provides a numeric representation for calculations and modeling.

### 4.7 Final Data Quality Check

In [17]:
print(f'Remaining rows : {df.shape[0]:,}')
print(f'Remaining columns : {df.shape[1]}')
print(f'Total missing : {df.isnull().sum().sum()}')
print(f'Duplicate records : {df.duplicated().sum()}')

Remaining rows : 7,032
Remaining columns : 22
Total missing : 0
Duplicate records : 0


### 4.8 Cleaning Summary

The following cleaning steps were performed:

- Identified 11 blank string values in `TotalCharges`.
- Converted `TotalCharges` from text to numeric.
- Verified that the affected records corresponded to customers with `tenure = 0`.
- Removed the 11 incomplete records.
- Created a numeric target variable (`Churn_Flag`) while preserving the original `Churn` column.

The dataset is now fully cleaned and ready for business-oriented exploratory analysis.